## Imports and project paths

In [1]:
from pathlib import Path

import cdsapi
import xarray as xr

In [2]:
project_root = Path.cwd().resolve()

while project_root != project_root.parent:
    if (project_root / "data").exists():
        break
    project_root = project_root.parent

weather_folder = (
    project_root
    / "data"
    / "raw"
    / "weather"
    / "era5_land"
)

weather_folder.mkdir(parents=True, exist_ok=True)

pilot_path = (
    weather_folder
    / "era5_land_manitoba_2024_07_01_03.nc"
)

print("Project root:", project_root)
print("Output file:", pilot_path)

Project root: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence
Output file: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/era5_land_manitoba_2024_07_01_03.nc


# Three-day pilot request

This requests:

+ Temperature
+ Dew-point temperature
+ East–west wind
+ North–south wind
+ Total precipitation

In [3]:
dataset = "reanalysis-era5-land"

request = {
    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "total_precipitation",
    ],
    "year": "2024",
    "month": "07",
    "day": [
        "01",
        "02",
        "03",
    ],
    "time": [
        "00:00", "01:00", "02:00", "03:00",
        "04:00", "05:00", "06:00", "07:00",
        "08:00", "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00", "15:00",
        "16:00", "17:00", "18:00", "19:00",
        "20:00", "21:00", "22:00", "23:00",
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",

    # Order: north, west, south, east
    "area": [
        60.2,
        -102.3,
        48.8,
        -89.5,
    ],
}

client = cdsapi.Client()

client.retrieve(
    dataset,
    request,
    str(pilot_path),
)

2026-07-22 13:40:34,326 INFO Request ID is ac5769de-62bd-48ba-b3df-0ece5c6f8080
2026-07-22 13:40:34,512 INFO status has been updated to accepted
2026-07-22 13:40:48,937 INFO status has been updated to running
2026-07-22 13:41:52,147 INFO status has been updated to successful


'/Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/era5_land_manitoba_2024_07_01_03.nc'

In [4]:
print("File exists:", pilot_path.exists())

if pilot_path.exists():
    print(
        "File size:",
        round(pilot_path.stat().st_size / 1_000_000, 2),
        "MB",
    )

File exists: True
File size: 7.92 MB


## NetCDF file

In [5]:
weather = xr.open_dataset(pilot_path)

weather

<xarray.Dataset> Size: 21MB
Dimensions:     (valid_time: 72, latitude: 115, longitude: 129)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 576B 2024-07-01 ... 2024-07-03T23...
    expver      (valid_time) <U4 1kB ...
  * latitude    (latitude) float64 920B 60.2 60.1 60.0 59.9 ... 49.0 48.9 48.8
  * longitude   (longitude) float64 1kB -102.3 -102.2 -102.1 ... -89.6 -89.5
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 4MB ...
    d2m         (valid_time, latitude, longitude) float32 4MB ...
    u10         (valid_time, latitude, longitude) float32 4MB ...
    v10         (valid_time, latitude, longitude) float32 4MB ...
    tp          (valid_time, latitude, longitude) float32 4MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-07-22T18:41 GRIB to CDM+CF via cfgrib-0.9.1...

In [6]:
print("Variables:")
print(list(weather.data_vars))

print("\nDimensions:")
print(dict(weather.sizes))

print("\nCoordinates:")
print(list(weather.coords))

Variables:
['t2m', 'd2m', 'u10', 'v10', 'tp']

Dimensions:
{'valid_time': 72, 'latitude': 115, 'longitude': 129}

Coordinates:
['number', 'valid_time', 'latitude', 'longitude', 'expver']


## Inspect units and values

In [7]:
for variable in weather.data_vars:
    data = weather[variable]

    print(f"\nVariable: {variable}")
    print("Long name:", data.attrs.get("long_name"))
    print("Units:", data.attrs.get("units"))
    print("Minimum:", float(data.min()))
    print("Maximum:", float(data.max()))
    print("Missing values:", int(data.isnull().sum()))


Variable: t2m
Long name: 2 metre temperature
Units: K
Minimum: 276.85205078125
Maximum: 302.816162109375
Missing values: 97056

Variable: d2m
Long name: 2 metre dewpoint temperature
Units: K
Minimum: 273.33837890625
Maximum: 292.868896484375
Missing values: 97056

Variable: u10
Long name: 10 metre U wind component
Units: m s**-1
Minimum: -8.993392944335938
Maximum: 9.962371826171875
Missing values: 97056

Variable: v10
Long name: 10 metre V wind component
Units: m s**-1
Minimum: -11.565719604492188
Maximum: 9.345001220703125
Missing values: 97056

Variable: tp
Long name: Total precipitation
Units: m
Minimum: 0.0
Maximum: 0.049592018127441406
Missing values: 97056


## Create useful hourly weather variables

In [8]:
import numpy as np
import pandas as pd

hourly_weather = xr.Dataset(
    {
        # Kelvin to degrees Celsius
        "TEMPERATURE_C": weather["t2m"] - 273.15,
        "DEWPOINT_C": weather["d2m"] - 273.15,

        # Wind-vector magnitude
        "WIND_SPEED_MS": np.sqrt(
            weather["u10"] ** 2
            + weather["v10"] ** 2
        ),

        # Metres of water to millimetres
        "PRECIPITATION_MM": weather["tp"] * 1000,
    }
)

# Estimate relative humidity from temperature and dew point.
hourly_weather["RELATIVE_HUMIDITY_PCT"] = (
    100
    * np.exp(
        (
            17.625 * hourly_weather["DEWPOINT_C"]
            / (243.04 + hourly_weather["DEWPOINT_C"])
        )
        -
        (
            17.625 * hourly_weather["TEMPERATURE_C"]
            / (243.04 + hourly_weather["TEMPERATURE_C"])
        )
    )
).clip(min=0, max=100)

hourly_weather

<xarray.Dataset> Size: 21MB
Dimensions:                (valid_time: 72, latitude: 115, longitude: 129)
Coordinates:
  * valid_time             (valid_time) datetime64[ns] 576B 2024-07-01 ... 20...
    expver                 (valid_time) <U4 1kB '0001' '0001' ... '0001' '0001'
  * latitude               (latitude) float64 920B 60.2 60.1 60.0 ... 48.9 48.8
  * longitude              (longitude) float64 1kB -102.3 -102.2 ... -89.6 -89.5
    number                 int64 8B 0
Data variables:
    TEMPERATURE_C          (valid_time, latitude, longitude) float32 4MB 20.9...
    DEWPOINT_C             (valid_time, latitude, longitude) float32 4MB 8.07...
    WIND_SPEED_MS          (valid_time, latitude, longitude) float32 4MB 4.33...
    PRECIPITATION_MM       (valid_time, latitude, longitude) float32 4MB 0.00...
    RELATIVE_HUMIDITY_PCT  (valid_time, latitude, longitude) float32 4MB 43.4...

In [9]:
for variable in hourly_weather.data_vars:
    print(
        variable,
        "min:",
        round(float(hourly_weather[variable].min()), 2),
        "max:",
        round(float(hourly_weather[variable].max()), 2),
    )

TEMPERATURE_C min: 3.7 max: 29.67
DEWPOINT_C min: 0.19 max: 19.72
WIND_SPEED_MS min: 0.01 max: 11.65
PRECIPITATION_MM min: 0.0 max: 49.59
RELATIVE_HUMIDITY_PCT min: 21.36 max: 100.0


## Convert UTC timestamps to Manitoba time

ERA5 timestamps use UTC, while the historical wildfire dates represent Manitoba calendar dates. We’ll convert the weather timestamps to America/Winnipeg.

In [10]:
utc_times = pd.DatetimeIndex(
    hourly_weather["valid_time"].values
).tz_localize("UTC")

local_times = utc_times.tz_convert(
    "America/Winnipeg"
)

# Calendar date for temperature, humidity and wind observations.
instantaneous_dates = (
    local_times
    .tz_localize(None)
    .normalize()
    .to_numpy(dtype="datetime64[ns]")
)

# Precipitation belongs to the hourly interval ending at its timestamp.
# Subtracting one nanosecond assigns midnight-ending precipitation
# to the preceding local calendar day.
precipitation_dates = (
    (local_times - pd.Timedelta(nanoseconds=1))
    .tz_localize(None)
    .normalize()
    .to_numpy(dtype="datetime64[ns]")
)

print("First UTC timestamp:", utc_times[0])
print("First Manitoba timestamp:", local_times[0])
print("Last Manitoba timestamp:", local_times[-1])

First UTC timestamp: 2024-07-01 00:00:00+00:00
First Manitoba timestamp: 2024-06-30 19:00:00-05:00
Last Manitoba timestamp: 2024-07-03 18:00:00-05:00


## Aggregate hourly weather into daily features

In [11]:
hourly_weather = hourly_weather.assign_coords(
    LOCAL_DATE=(
        "valid_time",
        instantaneous_dates,
    )
)

precipitation_hourly = (
    hourly_weather["PRECIPITATION_MM"]
    .assign_coords(
        PRECIPITATION_DATE=(
            "valid_time",
            precipitation_dates,
        )
    )
)

daily_precipitation = (
    precipitation_hourly
    .groupby("PRECIPITATION_DATE")
    .sum("valid_time")
    .rename(
        {"PRECIPITATION_DATE": "LOCAL_DATE"}
    )
)

daily_weather = xr.Dataset(
    {
        "TEMP_MAX_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .max("valid_time")
        ),
        "TEMP_MIN_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .min("valid_time")
        ),
        "TEMP_MEAN_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "RH_MIN_PCT": (
            hourly_weather["RELATIVE_HUMIDITY_PCT"]
            .groupby("LOCAL_DATE")
            .min("valid_time")
        ),
        "RH_MEAN_PCT": (
            hourly_weather["RELATIVE_HUMIDITY_PCT"]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "WIND_MAX_MS": (
            hourly_weather["WIND_SPEED_MS"]
            .groupby("LOCAL_DATE")
            .max("valid_time")
        ),
        "WIND_MEAN_MS": (
            hourly_weather["WIND_SPEED_MS"]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "PRECIPITATION_MM": daily_precipitation,
    }
)

daily_weather

<xarray.Dataset> Size: 2MB
Dimensions:           (latitude: 115, longitude: 129, LOCAL_DATE: 4)
Coordinates:
  * latitude          (latitude) float64 920B 60.2 60.1 60.0 ... 49.0 48.9 48.8
  * longitude         (longitude) float64 1kB -102.3 -102.2 ... -89.6 -89.5
  * LOCAL_DATE        (LOCAL_DATE) datetime64[ns] 32B 2024-06-30 ... 2024-07-03
    number            int64 8B 0
Data variables:
    TEMP_MAX_C        (LOCAL_DATE, latitude, longitude) float32 237kB 21.45 ....
    TEMP_MIN_C        (LOCAL_DATE, latitude, longitude) float32 237kB 16.88 ....
    TEMP_MEAN_C       (LOCAL_DATE, latitude, longitude) float32 237kB 19.97 ....
    RH_MIN_PCT        (LOCAL_DATE, latitude, longitude) float32 237kB 43.41 ....
    RH_MEAN_PCT       (LOCAL_DATE, latitude, longitude) float32 237kB 48.29 ....
    WIND_MAX_MS       (LOCAL_DATE, latitude, longitude) float32 237kB 4.338 ....
    WIND_MEAN_MS      (LOCAL_DATE, latitude, longitude) float32 237kB 3.399 ....
    PRECIPITATION_MM  (LOCAL_DATE, latitude, longitude) float32 237kB 0.00129...

## Keep only complete Manitoba days

Because the pilot begins and ends at UTC midnight, the first and last Manitoba dates are incomplete. July 1 and July 2 should each contain all 24 hours.

In [12]:
instantaneous_counts = pd.Series(
    instantaneous_dates
).value_counts()

precipitation_counts = pd.Series(
    precipitation_dates
).value_counts()

complete_dates = sorted(
    set(
        instantaneous_counts[
            instantaneous_counts == 24
        ].index
    )
    &
    set(
        precipitation_counts[
            precipitation_counts == 24
        ].index
    )
)

complete_dates = np.array(
    complete_dates,
    dtype="datetime64[ns]",
)

daily_weather_complete = daily_weather.sel(
    LOCAL_DATE=complete_dates
)

print("Complete local dates:")
print(daily_weather_complete["LOCAL_DATE"].values)

print("\nDaily dataset dimensions:")
print(dict(daily_weather_complete.sizes))

Complete local dates:
['2024-07-01T00:00:00.000000000' '2024-07-02T00:00:00.000000000']

Daily dataset dimensions:
{'latitude': 115, 'longitude': 129, 'LOCAL_DATE': 2}


## Inspect one sample location

In [13]:
sample_weather = (
    daily_weather_complete
    .sel(
        latitude=49.9,
        longitude=-97.1,
        method="nearest",
    )
    .to_dataframe()
)

display(sample_weather)

,number,latitude,longitude,TEMP_MAX_C,TEMP_MIN_C,TEMP_MEAN_C,RH_MIN_PCT,RH_MEAN_PCT,WIND_MAX_MS,WIND_MEAN_MS,PRECIPITATION_MM
LOCAL_DATE,,,,,,,,,,,
2024-07-01,0,49.9,-97.1,20.122467,13.851227,16.657715,63.092392,79.585945,5.957516,4.590501,134.964401
2024-07-02,0,49.9,-97.1,21.151031,16.184723,18.820267,73.325745,85.390053,6.199662,4.584035,295.024994


## Save the processed pilot

In [14]:
daily_pilot_path = (
    weather_folder
    / "era5_land_manitoba_daily_pilot_2024_07.nc"
)

daily_weather_complete.to_netcdf(
    daily_pilot_path
)

print("Saved:", daily_pilot_path)
print("File exists:", daily_pilot_path.exists())

Saved: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/era5_land_manitoba_daily_pilot_2024_07.nc
File exists: True


## Load the Manitoba grid reference

In [15]:
grid_reference_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_grid_reference_10km.parquet"
)

grid_reference = pd.read_parquet(
    grid_reference_path
)

print("Grid-reference rows:", len(grid_reference))
print("Duplicate GRID_ID values:", grid_reference["GRID_ID"].duplicated().sum())

display(grid_reference.head())

Grid-reference rows: 6501
Duplicate GRID_ID values: 0


,GRID_ID,ROW,COLUMN,CENTER_LATITUDE,CENTER_LONGITUDE,MB_AREA_KM2,MB_COVERAGE_PCT,FIRE_COUNT_2005_2025
0,MB_005_000,5,0,49.034036,-101.408339,14.489028,14.489028,0
1,MB_006_000,6,0,49.122970,-101.428824,2.229798,2.229798,0
2,MB_007_000,7,0,49.211928,-101.449396,7.044500,7.044500,0
3,MB_008_000,8,0,49.300912,-101.470056,0.002842,0.002842,0
4,MB_004_001,4,1,48.958440,-101.252939,4.265625,4.265625,0


## Open the processed daily pilot

Use the saved file rather than relying only on variables stored in memory:

In [16]:
daily_pilot_path = (
    project_root
    / "data"
    / "raw"
    / "weather"
    / "era5_land"
    / "era5_land_manitoba_daily_pilot_2024_07.nc"
)

daily_weather = xr.open_dataset(
    daily_pilot_path
)

print("Daily weather dimensions:")
print(dict(daily_weather.sizes))

print("\nDaily variables:")
print(list(daily_weather.data_vars))

print("\nLocal dates:")
print(daily_weather["LOCAL_DATE"].values)

Daily weather dimensions:
{'LOCAL_DATE': 2, 'latitude': 115, 'longitude': 129}

Daily variables:
['TEMP_MAX_C', 'TEMP_MIN_C', 'TEMP_MEAN_C', 'RH_MIN_PCT', 'RH_MEAN_PCT', 'WIND_MAX_MS', 'WIND_MEAN_MS', 'PRECIPITATION_MM']

Local dates:
['2024-07-01T00:00:00.000000000' '2024-07-02T00:00:00.000000000']


## Create latitude and longitude indexers

Both indexers use the same dimension, GRID_ID. This tells Xarray to retrieve one weather location for each Manitoba grid cell, instead of creating every possible latitude–longitude combination.

In [17]:
grid_ids = grid_reference["GRID_ID"].to_numpy()

latitude_indexer = xr.DataArray(
    grid_reference["CENTER_LATITUDE"].to_numpy(),
    dims="GRID_ID",
    coords={"GRID_ID": grid_ids},
)

longitude_indexer = xr.DataArray(
    grid_reference["CENTER_LONGITUDE"].to_numpy(),
    dims="GRID_ID",
    coords={"GRID_ID": grid_ids},
)

print("Latitude indexer:", latitude_indexer.shape)
print("Longitude indexer:", longitude_indexer.shape)

Latitude indexer: (6501,)
Longitude indexer: (6501,)


## Select the nearest ERA5-Land location

In [18]:
weather_at_grid = daily_weather.sel(
    latitude=latitude_indexer,
    longitude=longitude_indexer,
    method="nearest",
)

print("Weather assigned to grid dimensions:")
print(dict(weather_at_grid.sizes))

print("\nVariables:")
print(list(weather_at_grid.data_vars))

Weather assigned to grid dimensions:
{'LOCAL_DATE': 2, 'GRID_ID': 6501}

Variables:
['TEMP_MAX_C', 'TEMP_MIN_C', 'TEMP_MEAN_C', 'RH_MIN_PCT', 'RH_MEAN_PCT', 'WIND_MAX_MS', 'WIND_MEAN_MS', 'PRECIPITATION_MM']


## Examine matched ERA5 coordinates

Several Manitoba cells may use the same ERA5 point because the two grids are similar but not identical.

In [20]:
era5_matches = pd.DataFrame(
    {
        "GRID_ID": weather_at_grid["GRID_ID"].values,
        "ERA5_LATITUDE": weather_at_grid["latitude"].values,
        "ERA5_LONGITUDE": weather_at_grid["longitude"].values,
    }
)

print("Manitoba grid cells:", len(era5_matches))

print(
    "Unique ERA5 locations used:",
    len(
        era5_matches[
            ["ERA5_LATITUDE", "ERA5_LONGITUDE"]
        ].drop_duplicates()
    ),
)

display(era5_matches.head())

Manitoba grid cells: 6501
Unique ERA5 locations used: 6006


,GRID_ID,ERA5_LATITUDE,ERA5_LONGITUDE
0,MB_005_000,49.0,-101.4
1,MB_006_000,49.1,-101.4
2,MB_007_000,49.2,-101.4
3,MB_008_000,49.3,-101.5
4,MB_004_001,49.0,-101.3


## Convert the weather to a table

In [21]:
weather_grid_df = (
    weather_at_grid
    .to_dataframe()
    .reset_index()
)

weather_grid_df = weather_grid_df.rename(
    columns={
        "latitude": "ERA5_LATITUDE",
        "longitude": "ERA5_LONGITUDE",
    }
)

weather_grid_df["LOCAL_DATE"] = (
    pd.to_datetime(weather_grid_df["LOCAL_DATE"])
    .dt.normalize()
)

print("Weather-table rows:", len(weather_grid_df))
print("Weather-table columns:", weather_grid_df.columns.tolist())

display(weather_grid_df.head())

Weather-table rows: 13002
Weather-table columns: ['LOCAL_DATE', 'GRID_ID', 'TEMP_MAX_C', 'TEMP_MIN_C', 'TEMP_MEAN_C', 'RH_MIN_PCT', 'RH_MEAN_PCT', 'WIND_MAX_MS', 'WIND_MEAN_MS', 'PRECIPITATION_MM', 'number', 'ERA5_LATITUDE', 'ERA5_LONGITUDE']


,LOCAL_DATE,GRID_ID,TEMP_MAX_C,TEMP_MIN_C,TEMP_MEAN_C,RH_MIN_PCT,RH_MEAN_PCT,WIND_MAX_MS,WIND_MEAN_MS,PRECIPITATION_MM,number,ERA5_LATITUDE,ERA5_LONGITUDE
0,2024-07-01,MB_005_000,23.954254,14.601227,18.988199,67.112061,76.911171,5.686960,3.767682,34.090298,0,49.0,-101.4
1,2024-07-01,MB_006_000,23.879059,14.482086,18.875977,67.035118,76.986351,5.569101,3.668532,54.311123,0,49.1,-101.4
2,2024-07-01,MB_007_000,23.756012,14.347321,18.728027,67.043526,77.292221,5.392048,3.525611,92.186432,0,49.2,-101.4
3,2024-07-01,MB_008_000,23.771637,14.048492,18.487875,66.893036,77.956306,5.313594,3.486224,110.641068,0,49.3,-101.5
4,2024-07-01,MB_004_001,23.921051,14.657867,19.032959,67.247284,76.792847,5.724463,3.845844,35.343861,0,49.0,-101.3


## Add the original grid-centre coordinates

In [22]:
grid_location_columns = [
    "GRID_ID",
    "CENTER_LATITUDE",
    "CENTER_LONGITUDE",
    "MB_AREA_KM2",
    "MB_COVERAGE_PCT",
]

weather_grid_df = weather_grid_df.merge(
    grid_reference[grid_location_columns],
    on="GRID_ID",
    how="left",
    validate="many_to_one",
)

print("Rows after merge:", len(weather_grid_df))
print(
    "Missing grid centres:",
    weather_grid_df["CENTER_LATITUDE"].isna().sum(),
)

Rows after merge: 13002
Missing grid centres: 0


## Validate weather values

In [23]:
weather_variables = [
    "TEMP_MAX_C",
    "TEMP_MIN_C",
    "TEMP_MEAN_C",
    "RH_MIN_PCT",
    "RH_MEAN_PCT",
    "WIND_MAX_MS",
    "WIND_MEAN_MS",
    "PRECIPITATION_MM",
]

expected_rows = (
    len(grid_reference)
    * daily_weather.sizes["LOCAL_DATE"]
)

print("Expected rows:", expected_rows)
print("Actual rows:", len(weather_grid_df))

print(
    "Duplicate date-grid rows:",
    weather_grid_df.duplicated(
        subset=["LOCAL_DATE", "GRID_ID"]
    ).sum(),
)

print("\nMissing weather values:")
display(
    weather_grid_df[
        weather_variables
    ].isna().sum().to_frame("missing_count")
)

print("\nWeather summaries:")
display(
    weather_grid_df[
        weather_variables
    ].describe().T
)

Expected rows: 13002
Actual rows: 13002
Duplicate date-grid rows: 0

Missing weather values:


,missing_count
TEMP_MAX_C,88
TEMP_MIN_C,88
TEMP_MEAN_C,88
RH_MIN_PCT,88
RH_MEAN_PCT,88
WIND_MAX_MS,88
WIND_MEAN_MS,88
PRECIPITATION_MM,0



Weather summaries:


,count,mean,std,min,25%,50%,75%,max
TEMP_MAX_C,12914.0,20.831585,4.048451,13.342194,17.601288,20.342682,24.269196,29.666168
TEMP_MIN_C,12914.0,14.113977,1.204436,6.891754,13.284821,14.016876,14.987946,17.597565
TEMP_MEAN_C,12914.0,17.379061,2.384090,12.912715,15.572312,17.079161,19.296291,22.909180
RH_MIN_PCT,12914.0,58.567406,20.323654,23.715736,37.925362,62.065323,72.826134,99.397209
RH_MEAN_PCT,12914.0,73.870560,16.230587,39.935337,58.227913,78.329330,86.520485,99.959595
WIND_MAX_MS,12914.0,4.666485,1.787422,1.816193,3.300685,4.127761,5.843566,11.297716
WIND_MEAN_MS,12914.0,3.297650,1.434469,0.965799,2.275848,2.930097,4.120981,8.819597
PRECIPITATION_MM,13002.0,75.993416,116.701378,0.000000,0.104884,14.947676,92.695633,643.208740


## Add the wildfire target

In [39]:
targets_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_daily_fire_targets_2005_2025.parquet"
)

fire_targets = pd.read_parquet(
    targets_path
)

fire_targets["FIRE_DATE"] = (
    pd.to_datetime(fire_targets["FIRE_DATE"])
    .dt.normalize()
)

pilot_model_table = weather_grid_df.merge(
    fire_targets[
        [
            "FIRE_DATE",
            "GRID_ID",
            "FIRE_OCCURRED",
            "FIRE_COUNT",
            "NATURAL_FIRE_COUNT",
            "HUMAN_FIRE_COUNT",
        ]
    ],
    left_on=["LOCAL_DATE", "GRID_ID"],
    right_on=["FIRE_DATE", "GRID_ID"],
    how="left",
    validate="one_to_one",
)

target_columns = [
    "FIRE_OCCURRED",
    "FIRE_COUNT",
    "NATURAL_FIRE_COUNT",
    "HUMAN_FIRE_COUNT",
]

pilot_model_table[target_columns] = (
    pilot_model_table[target_columns]
    .fillna(0)
    .astype("int16")
)

pilot_model_table = pilot_model_table.drop(
    columns="FIRE_DATE"
)

print("Pilot modelling rows:", len(pilot_model_table))
print(
    "Positive grid-cell days:",
    pilot_model_table["FIRE_OCCURRED"].sum(),
)
print(
    "Negative grid-cell days:",
    (pilot_model_table["FIRE_OCCURRED"] == 0).sum(),
)
print(
    "Duplicate date-grid rows:",
    pilot_model_table.duplicated(
        subset=["LOCAL_DATE", "GRID_ID"]
    ).sum(),
)

Pilot modelling rows: 13002
Positive grid-cell days: 3
Negative grid-cell days: 12999
Duplicate date-grid rows: 0


In [25]:
weather_grid_output = (
    project_root
    / "data"
    / "interim"
    / "era5_grid_weather_pilot_2024_07.parquet"
)

model_pilot_output = (
    project_root
    / "data"
    / "interim"
    / "manitoba_model_table_pilot_2024_07.parquet"
)

weather_grid_df.to_parquet(
    weather_grid_output,
    index=False,
)

pilot_model_table.to_parquet(
    model_pilot_output,
    index=False,
)

print("Weather table saved:", weather_grid_output)
print("Weather file exists:", weather_grid_output.exists())

print("\nModel pilot saved:", model_pilot_output)
print("Model file exists:", model_pilot_output.exists())

Weather table saved: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/interim/era5_grid_weather_pilot_2024_07.parquet
Weather file exists: True

Model pilot saved: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/interim/manitoba_model_table_pilot_2024_07.parquet
Model file exists: True


## Identify the affected grid cells

In [32]:
instantaneous_variables = [
    "TEMP_MAX_C",
    "TEMP_MIN_C",
    "TEMP_MEAN_C",
    "RH_MIN_PCT",
    "RH_MEAN_PCT",
    "WIND_MAX_MS",
    "WIND_MEAN_MS",
]

missing_weather_mask = (
    weather_grid_df[instantaneous_variables]
    .isna()
    .any(axis=1)
)

missing_weather_cells = (
    weather_grid_df.loc[
        missing_weather_mask,
        [
            "GRID_ID",
            "CENTER_LATITUDE",
            "CENTER_LONGITUDE",
            "ERA5_LATITUDE",
            "ERA5_LONGITUDE",
            "MB_AREA_KM2",
            "MB_COVERAGE_PCT",
        ],
    ]
    .drop_duplicates("GRID_ID")
    .sort_values("MB_COVERAGE_PCT")
)

print("Rows with missing weather:", missing_weather_mask.sum())
print("Unique affected grid cells:", len(missing_weather_cells))

display(missing_weather_cells.head(20))

Rows with missing weather: 0
Unique affected grid cells: 0


,GRID_ID,CENTER_LATITUDE,CENTER_LONGITUDE,ERA5_LATITUDE,ERA5_LONGITUDE,MB_AREA_KM2,MB_COVERAGE_PCT


In [33]:
valid_era5_mask = (
    daily_weather["TEMP_MEAN_C"]
    .notnull()
    .all("LOCAL_DATE")
)

valid_era5_points = (
    valid_era5_mask
    .stack(ERA5_POINT=("latitude", "longitude"))
)

valid_era5_points = valid_era5_points.where(
    valid_era5_points,
    drop=True,
)

valid_latitudes = valid_era5_points["latitude"].values
valid_longitudes = valid_era5_points["longitude"].values

print("Valid ERA5-Land locations:", len(valid_era5_points))

Valid ERA5-Land locations: 13487


## Match each cell to its nearest valid land point

In [34]:
import numpy as np

from sklearn.neighbors import BallTree

EARTH_RADIUS_KM = 6371.0088

valid_coordinates_radians = np.radians(
    np.column_stack(
        [
            valid_latitudes,
            valid_longitudes,
        ]
    )
)

grid_coordinates_radians = np.radians(
    grid_reference[
        [
            "CENTER_LATITUDE",
            "CENTER_LONGITUDE",
        ]
    ].to_numpy()
)

tree = BallTree(
    valid_coordinates_radians,
    metric="haversine",
)

distance_radians, nearest_indices = tree.query(
    grid_coordinates_radians,
    k=1,
)

nearest_indices = nearest_indices[:, 0]

nearest_latitudes = valid_latitudes[nearest_indices]
nearest_longitudes = valid_longitudes[nearest_indices]

match_distances_km = (
    distance_radians[:, 0]
    * EARTH_RADIUS_KM
)

era5_match_table = grid_reference[
    [
        "GRID_ID",
        "CENTER_LATITUDE",
        "CENTER_LONGITUDE",
        "MB_AREA_KM2",
        "MB_COVERAGE_PCT",
    ]
].copy()

era5_match_table["ERA5_LATITUDE"] = nearest_latitudes
era5_match_table["ERA5_LONGITUDE"] = nearest_longitudes
era5_match_table["ERA5_MATCH_DISTANCE_KM"] = match_distances_km

display(
    era5_match_table[
        "ERA5_MATCH_DISTANCE_KM"
    ].describe()
)

count    6501.000000
mean        3.485795
std         1.608817
min         0.027308
25%         2.392972
50%         3.441408
75%         4.599427
max        35.030632
Name: ERA5_MATCH_DISTANCE_KM, dtype: float64

## Extract weather again using valid points

In [35]:
grid_ids = era5_match_table["GRID_ID"].to_numpy()

valid_latitude_indexer = xr.DataArray(
    era5_match_table["ERA5_LATITUDE"].to_numpy(),
    dims="GRID_ID",
    coords={"GRID_ID": grid_ids},
)

valid_longitude_indexer = xr.DataArray(
    era5_match_table["ERA5_LONGITUDE"].to_numpy(),
    dims="GRID_ID",
    coords={"GRID_ID": grid_ids},
)

weather_at_grid_valid = daily_weather.sel(
    latitude=valid_latitude_indexer,
    longitude=valid_longitude_indexer,
)

print(dict(weather_at_grid_valid.sizes))

{'LOCAL_DATE': 2, 'GRID_ID': 6501}


## Rebuild and validate the weather table

In [36]:
weather_grid_df = (
    weather_at_grid_valid
    .to_dataframe()
    .reset_index()
    .rename(
        columns={
            "latitude": "ERA5_LATITUDE",
            "longitude": "ERA5_LONGITUDE",
        }
    )
)

weather_grid_df["LOCAL_DATE"] = (
    pd.to_datetime(weather_grid_df["LOCAL_DATE"])
    .dt.normalize()
)

weather_grid_df = weather_grid_df.merge(
    era5_match_table[
        [
            "GRID_ID",
            "CENTER_LATITUDE",
            "CENTER_LONGITUDE",
            "MB_AREA_KM2",
            "MB_COVERAGE_PCT",
            "ERA5_MATCH_DISTANCE_KM",
        ]
    ],
    on="GRID_ID",
    how="left",
    validate="many_to_one",
)

print("Weather rows:", len(weather_grid_df))

print(
    "Duplicate date-grid rows:",
    weather_grid_df.duplicated(
        subset=["LOCAL_DATE", "GRID_ID"]
    ).sum(),
)

print("\nMissing weather values:")

display(
    weather_grid_df[
        weather_variables
    ]
    .isna()
    .sum()
    .to_frame("missing_count")
)

print(
    "Maximum ERA5 matching distance:",
    round(
        weather_grid_df[
            "ERA5_MATCH_DISTANCE_KM"
        ].max(),
        2,
    ),
    "km",
)

Weather rows: 13002
Duplicate date-grid rows: 0

Missing weather values:


,missing_count
TEMP_MAX_C,0
TEMP_MIN_C,0
TEMP_MEAN_C,0
RH_MIN_PCT,0
RH_MEAN_PCT,0
WIND_MAX_MS,0
WIND_MEAN_MS,0
PRECIPITATION_MM,0


Maximum ERA5 matching distance: 35.03 km


## Save the reusable ERA5 match table

This cell-to-ERA5 mapping can be reused for every year.

In [40]:
era5_match_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_grid_to_era5_land_matches.parquet"
)

era5_match_table.to_parquet(
    era5_match_path,
    index=False,
)

print("ERA5 match table saved:", era5_match_path)
print("File exists:", era5_match_path.exists())

ERA5 match table saved: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/interim/manitoba_grid_to_era5_land_matches.parquet
File exists: True


## Verification of corrected results

In [41]:
print("Weather rows:", len(weather_grid_df))

print(
    "Missing weather values:",
    weather_grid_df[weather_variables].isna().sum().sum(),
)

print(
    "Positive grid-cell days:",
    pilot_model_table["FIRE_OCCURRED"].sum(),
)

print(
    "Duplicate date-grid rows:",
    pilot_model_table.duplicated(
        subset=["LOCAL_DATE", "GRID_ID"]
    ).sum(),
)

print(
    "Maximum ERA5 match distance:",
    round(
        weather_grid_df["ERA5_MATCH_DISTANCE_KM"].max(),
        2,
    ),
    "km",
)

Weather rows: 13002
Missing weather values: 0
Positive grid-cell days: 3
Duplicate date-grid rows: 0
Maximum ERA5 match distance: 35.03 km
